https://gml.noaa.gov/ccgg/mbl/mbl.html
- Uncertainty is 'monthly residual standard deviation'


In [ ]:
import pandas as pd
import numpy as np
import datetime
import xarray as xr

In [ ]:
# MAIN PROCESSING FUNCTION
# filetag = '2014-2018' 
# filetag = '2019-2023'

tags = ['2014-2018', '2019-2023']
combined = []
save = True

for filetag in tags:
    ### IMPORT 
    filepath = '../data/marineboundarylayer/co2_mbl_' + filetag + '_data.txt'
    df = pd.read_csv(filepath, sep="       ", header=None)

    old_columns = df.columns
    new_columns = ['decimalyr']

    # Columns
    sin_lat = np.array([-1.00, -0.95, -0.90, -0.85, -0.80, 
                        -0.75, -0.70, -0.65, -0.60, -0.55, -0.50])
    lat_deg = np.degrees(np.arcsin(sin_lat))

    for x in sin_lat:
        new_columns.append(str(np.round(x,3))+'mean')
        new_columns.append(str(np.round(x,3))+'sdev')

    df.rename({k:v for k,v in zip(old_columns, new_columns)}, axis=1, inplace=True)

    # Prepare arrays for mean and sdev
    mean_data = np.column_stack([df[f"{slat}mean"].values for slat in sin_lat])
    sdev_data = np.column_stack([df[f"{slat}sdev"].values for slat in sin_lat])

    # Build xarray Dataset
    ds = xr.Dataset(
        {
            "mean": (("decimalyr", "sin_lat"), mean_data),
            "sdev": (("decimalyr", "sin_lat"), sdev_data)
        },
        coords={
            "decimalyr": df["decimalyr"].values,
            "sin_lat": sin_lat
        }
    )

    combined.append(ds)


full_ds = xr.merge(combined).assign_coords(lat_deg=("sin_lat", np.degrees(np.arcsin(ds.sin_lat.values))))
full_ds.attrs['description']= 'NOAA Marine Boundary Layer CO2 data (surface) from 2014 to 2023'
full_ds.attrs['source'] = 'https://gml.noaa.gov/ccgg/mbl/data.php'
full_ds.attrs['date_accessed'] = datetime.datetime.now().strftime("%Y-%m-%d")

if save:
    # df.to_csv('../data/marineboundarylayer/co2_mbl_' + filetag + '_data_tabular.csv')
    # ds.to_netcdf('../data/marineboundarylayer/co2_mbl_' + filetag + '_data.nc')
    full_ds.to_netcdf('../data/marineboundarylayer/co2_mbl_2014-2023_dataset_combined.nc')

/var/folders/nt/sjynqxjj7cz9r15fkd5d4r_40000gp/T/ipykernel_8983/2336035967.py:11: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  df = pd.read_csv(filepath, sep="       ", header=None)
/var/folders/nt/sjynqxjj7cz9r15fkd5d4r_40000gp/T/ipykernel_8983/2336035967.py:11: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  df = pd.read_csv(filepath, sep="       ", header=None)


# outdated dev

In [59]:
mean_data

array([[406.042, 405.949, 405.837, ..., 406.009, 406.316, 406.675],
       [405.953, 405.859, 405.744, ..., 406.028, 406.366, 406.752],
       [405.873, 405.777, 405.662, ..., 406.039, 406.405, 406.814],
       ...,
       [417.813, 417.759, 417.686, ..., 417.913, 418.206, 418.551],
       [417.792, 417.735, 417.659, ..., 417.996, 418.338, 418.738],
       [417.57 , 417.541, 417.504, ..., 418.106, 418.48 , 418.917]])

In [ ]:
full_ds

<xarray.Dataset>
Dimensions:    (decimalyr: 481, sin_lat: 11)
Coordinates:
  * decimalyr  (decimalyr) float64 2.014e+03 2.014e+03 ... 2.024e+03 2.024e+03
  * sin_lat    (sin_lat) float64 -1.0 -0.95 -0.9 -0.85 ... -0.65 -0.6 -0.55 -0.5
    lat_deg    (sin_lat) float64 -90.0 -71.81 -64.16 ... -36.87 -33.37 -30.0
Data variables:
    mean       (decimalyr, sin_lat) float64 394.0 394.0 393.9 ... 418.5 418.9
    sdev       (decimalyr, sin_lat) float64 0.075 0.065 0.063 ... 0.423 0.359
Attributes:
    description:    NOAA Marine Boundary Layer CO2 data (surface) from 2014 t...
    source:         https://gml.noaa.gov/ccgg/mbl/data.php
    date_accessed:  2025-08-12

In [61]:
full_ds.lat_deg.values

array([-90.        , -71.80512766, -64.15806724, -58.21166938,
       -53.13010235, -48.59037789, -44.427004  , -40.54160187,
       -36.86989765, -33.36701297, -30.        ])

In [27]:
df.to_csv('../data/marineboundarylayer/co2_mbl_2014-2018_data_tabular.csv')

In [34]:
# Prepare arrays for mean and sdev
mean_data = np.column_stack([df[f"{slat}mean"].values for slat in sin_lat])
sdev_data = np.column_stack([df[f"{slat}sdev"].values for slat in sin_lat])

# Build xarray Dataset
ds = xr.Dataset(
    {
        "mean": (("decimalyr", "sin_lat"), mean_data),
        "sdev": (("decimalyr", "sin_lat"), sdev_data)
    },
    coords={
        "decimalyr": df["decimalyr"].values,
        "sin_lat": sin_lat
    }
)

In [35]:
ds

<xarray.Dataset>
Dimensions:    (decimalyr: 241, sin_lat: 11)
Coordinates:
  * decimalyr  (decimalyr) float64 2.014e+03 2.014e+03 ... 2.019e+03 2.019e+03
  * sin_lat    (sin_lat) float64 -1.0 -0.95 -0.9 -0.85 ... -0.65 -0.6 -0.55 -0.5
Data variables:
    mean       (decimalyr, sin_lat) float64 394.0 394.0 393.9 ... 406.3 406.7
    sdev       (decimalyr, sin_lat) float64 0.075 0.065 0.063 ... 0.325 0.28

In [ ]:
# SAVE TABLE FOR CONVERSION FROM SIN LATITUDE TO DEGREES SOUTH
sin_lat = np.array([-1.00, -0.95, -0.90, -0.85, -0.80, 
                    -0.75, -0.70, -0.65, -0.60, -0.55, -0.50])

# arcsine (in radians) → convert to degrees
lat_deg = np.degrees(np.arcsin(sin_lat))

# To save: 
pd.DataFrame(index=sin_lat, data=lat_deg, columns=['degS']).to_csv()

# To read in
# pd.read_csv('../data/marineboundarylayer/sine_lat_degreeS_table.csv', index_col=0)

In [28]:
pd.read_csv('../data/marineboundarylayer/co2_mbl_2014-2018_data_tabular.csv', index_col=0)

,decimalyr,-1.0mean,-1.0sdev,-0.95mean,-0.95sdev,-0.9mean,-0.9sdev,-0.85mean,-0.85sdev,-0.8mean,...,-0.7mean,-0.7sdev,-0.65mean,-0.65sdev,-0.6mean,-0.6sdev,-0.55mean,-0.55sdev,-0.5mean,-0.5sdev
0,2014.000000,394.026,0.075,393.989,0.065,393.941,0.063,393.888,0.075,393.841,...,393.828,0.126,393.886,0.137,393.999,0.147,394.173,0.156,394.409,0.161
1,2014.020833,393.944,0.095,393.903,0.086,393.853,0.082,393.802,0.088,393.765,...,393.796,0.128,393.883,0.135,394.027,0.143,394.229,0.150,394.487,0.154
2,2014.041667,393.863,0.112,393.818,0.104,393.767,0.100,393.720,0.103,393.695,...,393.766,0.128,393.879,0.133,394.048,0.138,394.273,0.143,394.549,0.147
3,2014.062500,393.787,0.124,393.741,0.118,393.690,0.114,393.648,0.114,393.632,...,393.736,0.127,393.869,0.131,394.058,0.133,394.301,0.136,394.591,0.140
4,2014.083333,393.723,0.131,393.676,0.125,393.626,0.121,393.588,0.120,393.580,...,393.707,0.125,393.854,0.126,394.056,0.127,394.311,0.131,394.611,0.133
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
236,2018.916667,406.336,0.085,406.273,0.076,406.189,0.084,406.093,0.117,405.998,...,405.896,0.271,405.929,0.304,406.035,0.315,406.216,0.300,406.459,0.263
237,2018.937500,406.274,0.105,406.202,0.095,406.111,0.100,406.007,0.130,405.910,...,405.821,0.288,405.871,0.322,406.002,0.333,406.213,0.317,406.488,0.278
238,2018.958333,406.219,0.096,406.134,0.093,406.030,0.104,405.918,0.138,405.819,...,405.753,0.298,405.828,0.332,405.989,0.340,406.231,0.324,406.536,0.283
239,2018.979167,406.133,0.106,406.044,0.102,405.935,0.115,405.821,0.150,405.725,...,405.696,0.308,405.801,0.338,405.994,0.344,406.269,0.324,406.602,0.282


In [21]:
df

,decimalyr,-1.0mean,-1.0sdev,-0.95mean,-0.95sdev,-0.9mean,-0.9sdev,-0.85mean,-0.85sdev,-0.8mean,...,-0.7mean,-0.7sdev,-0.65mean,-0.65sdev,-0.6mean,-0.6sdev,-0.55mean,-0.55sdev,-0.5mean,-0.5sdev
0,2014.000000,394.026,0.075,393.989,0.065,393.941,0.063,393.888,0.075,393.841,...,393.828,0.126,393.886,0.137,393.999,0.147,394.173,0.156,394.409,0.161
1,2014.020833,393.944,0.095,393.903,0.086,393.853,0.082,393.802,0.088,393.765,...,393.796,0.128,393.883,0.135,394.027,0.143,394.229,0.150,394.487,0.154
2,2014.041667,393.863,0.112,393.818,0.104,393.767,0.100,393.720,0.103,393.695,...,393.766,0.128,393.879,0.133,394.048,0.138,394.273,0.143,394.549,0.147
3,2014.062500,393.787,0.124,393.741,0.118,393.690,0.114,393.648,0.114,393.632,...,393.736,0.127,393.869,0.131,394.058,0.133,394.301,0.136,394.591,0.140
4,2014.083333,393.723,0.131,393.676,0.125,393.626,0.121,393.588,0.120,393.580,...,393.707,0.125,393.854,0.126,394.056,0.127,394.311,0.131,394.611,0.133
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
236,2018.916667,406.336,0.085,406.273,0.076,406.189,0.084,406.093,0.117,405.998,...,405.896,0.271,405.929,0.304,406.035,0.315,406.216,0.300,406.459,0.263
237,2018.937500,406.274,0.105,406.202,0.095,406.111,0.100,406.007,0.130,405.910,...,405.821,0.288,405.871,0.322,406.002,0.333,406.213,0.317,406.488,0.278
238,2018.958333,406.219,0.096,406.134,0.093,406.030,0.104,405.918,0.138,405.819,...,405.753,0.298,405.828,0.332,405.989,0.340,406.231,0.324,406.536,0.283
239,2018.979167,406.133,0.106,406.044,0.102,405.935,0.115,405.821,0.150,405.725,...,405.696,0.308,405.801,0.338,405.994,0.344,406.269,0.324,406.602,0.282


In [15]:
[np.round(x,3) for x in lat_deg]

[-90.0,
 -71.805,
 -64.158,
 -58.212,
 -53.13,
 -48.59,
 -44.427,
 -40.542,
 -36.87,
 -33.367,
 -30.0]